# Poliedros convexos

**Módulo 06 — Geometria Espacial** · Notebook 07 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0607a_poliedros_convexos.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (diedros entre faces) e [`0606a_triedros_e_angulos_poliedricos.ipynb`](0606a_triedros_e_angulos_poliedricos.ipynb) (ângulos poliédricos convexos, polígonos regulares num vértice, defeito angular)
- Módulo 04 — Geometria Plana: [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb) (soma dos ângulos internos de um polígono) e [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e seus ângulos internos)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- identificar poliedros convexos e contar seus vértices, arestas e faces;
- aplicar a Relação de Euler e a soma dos ângulos das faces para descobrir elementos desconhecidos;
- reconhecer planificações e decidir se uma figura plana fecha um sólido;
- explicar por que existem exatamente cinco poliedros de Platão e cinco poliedros regulares.

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets

## Por que os dados de RPG têm esses formatos?

Quem joga RPG de mesa conhece o kit: dados de **4, 6, 8, 12 e 20 faces** (chamados d4, d6, d8, d12 e d20). Repare que, em cada um deles, todas as faces são polígonos regulares iguais, e todos os cantos são iguais. É isso que torna o dado **justo**: nenhuma face tem vantagem sobre as outras.

Agora pense: por que não existe um dado assim com 10 faces? Ou 14? Ou 100?

(O d10 que aparece em alguns kits existe, mas as faces dele não são polígonos regulares: são "pipas", com dois lados maiores e dois menores.)

🕹️ Escolha um dado abaixo e gire-o com o mouse.

In [2]:
# Para cada dado: o sólido, o polígono das faces e a cor do desenho.
DADOS = {
    "d4 (tetraedro)": ("tetraedro", "triângulo equilátero", u3d.CORES["plano"]),
    "d6 (cubo)": ("cubo", "quadrado", u3d.CORES["plano_secundario"]),
    "d8 (octaedro)": ("octaedro", "triângulo equilátero", u3d.CORES["plano_terciario"]),
    "d12 (dodecaedro)": ("dodecaedro", "pentágono regular", u3d.CORES["destaque"]),
    "d20 (icosaedro)": ("icosaedro", "triângulo equilátero", u3d.CORES["solido"]),
}


def mostrar_dado(dado: str) -> None:
    """Desenha o dado escolhido e diz quantas faces ele tem."""
    solido, poligono, cor = DADOS[dado]
    vertices, faces = u3d.poliedro_de_platao(solido)
    fig = u3d.criar_figura(titulo=dado)
    fig = u3d.desenhar_poliedro(fig, vertices, faces, cor=cor, opacidade=0.7)
    u3d.mostrar(fig)
    print(f"Número de faces: {len(faces)}")
    print(f"Cada face é um {poligono}.")


interact(
    mostrar_dado,
    dado=widgets.Dropdown(options=list(DADOS), description="dado"),
);

interactive(children=(Dropdown(description='dado', options=('d4 (tetraedro)', 'd6 (cubo)', 'd8 (octaedro)', 'd…

Neste notebook você vai provar que esses cinco formatos são os **únicos** possíveis: não existe nenhum outro sólido convexo com todas as faces regulares iguais e todos os cantos iguais. No caminho, você vai conhecer uma das fórmulas mais bonitas da Matemática, a **Relação de Euler**, que relaciona os vértices, as arestas e as faces de qualquer poliedro convexo.

## 1. Poliedros e poliedros convexos

> 📌 **Definição.** Um **poliedro** é um sólido limitado por um número finito de polígonos (as **faces**), de modo que:
> - cada lado de uma face é lado de **exatamente duas** faces;
> - duas faces com um lado em comum não estão no mesmo plano.

**Elementos**
- **Faces:** os polígonos.
- **Arestas:** os lados das faces (cada aresta é comum a duas faces).
- **Vértices:** os vértices das faces. Em cada vértice há um **ângulo poliédrico** (notebook 06).

Vamos chamar de $V$, $A$ e $F$ os números de vértices, arestas e faces.

> 📌 **Definição.** Um poliedro é **convexo** quando o plano de **cada** face deixa todo o resto do poliedro de um mesmo lado.

Um jeito intuitivo: um poliedro é convexo se ele não tem "buracos" nem "afundamentos". Uma caixa de sapatos é convexa; uma estrela de Natal 3D, não.

**Nomes pelo número de faces:** tetraedro (4), pentaedro (5), hexaedro (6), heptaedro (7), octaedro (8), dodecaedro (12), icosaedro (20).

**Contando com Python.** Vamos guardar um poliedro como uma lista de vértices e uma lista de faces (cada face é uma lista de **números de vértices**). Contar vértices e faces é fácil: é o tamanho de cada lista. As arestas dão mais trabalho, porque cada uma aparece em **duas** faces. A solução é guardá-las num **conjunto** (`set`), que ignora repetições.

In [3]:
def contar_elementos(vertices: list, faces: list) -> tuple[int, int, int]:
    """Conta vértices (V), arestas (A) e faces (F) de um poliedro."""
    arestas = set()
    for face in faces:
        for k in range(len(face)):
            i = face[k]
            j = face[(k + 1) % len(face)]  # o próximo vértice (o último se liga ao primeiro)
            arestas.add(tuple(sorted((i, j))))  # (2, 5) e (5, 2) são a mesma aresta
    return len(vertices), len(arestas), len(faces)


# Exemplo: o cubo. Os vértices são pontos (x, y, z), usados só para desenhar; as
# faces são listas de NÚMEROS de vértices (a posição de cada um na lista).
vertices, faces = u3d.poliedro_de_platao("cubo")
print("As três primeiras faces:", faces[:3])
V, A, F = contar_elementos(vertices, faces)
print(f"Cubo: V = {V}, A = {A}, F = {F}")

# O desenho mostra o número de cada vértice, para você conferir as faces acima.
fig = u3d.criar_figura(titulo="O cubo, com os vértices numerados")
fig = u3d.desenhar_poliedro(
    fig, vertices, faces, opacidade=0.3, rotulos=[str(i) for i in range(V)]
)
u3d.mostrar(fig)

As três primeiras faces: [[1, 0, 3, 2], [4, 0, 1, 5], [3, 0, 4, 7]]
Cubo: V = 8, A = 12, F = 6


### Convexo ou não convexo?

🕹️ Escolha um poliedro e uma das suas faces. A face escolhida aparece em destaque, com o seu plano. Se o poliedro for convexo, **todo** o resto dele fica de um mesmo lado desse plano, qualquer que seja a face escolhida.

In [4]:
POLIEDROS_DA_SECAO_1 = {
    "Cubo": u3d.poliedro_de_platao("cubo"),
    "Octaedro": u3d.poliedro_de_platao("octaedro"),
    "Icosaedro": u3d.poliedro_de_platao("icosaedro"),
    "Cubo com cavidade": u3d.poliedro_exemplo("cubo_com_cavidade"),
    "Moldura": u3d.poliedro_exemplo("moldura"),
}
# Nos não convexos, uma face que mostra o problema.
SUGESTOES = {
    "Cubo com cavidade": "uma das faces da cavidade (faces 5 a 8)",
    "Moldura": "uma das faces de dentro do furo (faces 12 a 15)",
}


def testar_convexidade(poliedro: str, face_escolhida: int) -> None:
    """Destaca uma face e o seu plano, e diz se o poliedro é convexo."""
    vertices, faces = POLIEDROS_DA_SECAO_1[poliedro]
    # O resto da divisão mantém o número da face entre 0 e F − 1.
    numero = face_escolhida % len(faces)
    face = [vertices[i] for i in faces[numero]]
    plano = u3d.plano_por_tres_pontos(face[0], face[1], face[2])

    fig = u3d.criar_figura(titulo=f"{poliedro}: face {numero}")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.35)
    fig = u3d.desenhar_poligono(fig, face, cor=u3d.CORES["destaque"], opacidade=0.8)
    fig = u3d.desenhar_plano(fig, *plano, opacidade=0.15, tamanho=7)
    u3d.mostrar(fig)

    V, A, F = contar_elementos(vertices, faces)
    print(f"V = {V}, A = {A}, F = {F}")
    print(f"Convexo? {u3d.poliedro_e_convexo(vertices, faces)}")
    if poliedro in SUGESTOES:
        print(f"Experimente {SUGESTOES[poliedro]}.")


interact(
    testar_convexidade,
    poliedro=widgets.Dropdown(options=list(POLIEDROS_DA_SECAO_1), description="poliedro"),
    face_escolhida=widgets.IntSlider(
        min=0, max=15, step=1, value=0, description="face", continuous_update=False
    ),
);

interactive(children=(Dropdown(description='poliedro', options=('Cubo', 'Octaedro', 'Icosaedro', 'Cubo com cav…

**Pare e pense**

1. Quantos vértices, arestas e faces tem uma caixa de sapatos?
2. Na "moldura", escolha uma das faces de dentro do furo. O plano dela deixa o poliedro todo de um lado só?

<details>
<summary>Ver respostas</summary>

1. $V = 8$, $A = 12$, $F = 6$.
2. Não: o plano de uma face interna atravessa a moldura, deixando partes dela dos dois lados. Por isso ela não é convexa.
</details>

## 2. A Relação de Euler

Observe a tabela abaixo, com vários poliedros convexos. Procure uma relação entre $V$, $A$ e $F$ que vale em **todas** as linhas.

In [5]:
cubo = u3d.poliedro_de_platao("cubo")
tetraedro = u3d.poliedro_de_platao("tetraedro")
POLIEDROS_CONVEXOS = {
    "tetraedro": tetraedro,
    "cubo": cubo,
    "octaedro": u3d.poliedro_de_platao("octaedro"),
    "dodecaedro": u3d.poliedro_de_platao("dodecaedro"),
    "icosaedro": u3d.poliedro_de_platao("icosaedro"),
    "cuboctaedro": u3d.truncar_poliedro(*cubo, 0.5),
    "tetraedro truncado": u3d.truncar_poliedro(*tetraedro, 1 / 3),
}

print(f"{'poliedro':<20}{'V':>5}{'A':>5}{'F':>5}")
for nome, (vertices, faces) in POLIEDROS_CONVEXOS.items():
    V, A, F = contar_elementos(vertices, faces)
    print(f"{nome:<20}{V:>5}{A:>5}{F:>5}")

poliedro                V    A    F
tetraedro               4    6    4
cubo                    8   12    6
octaedro                6   12    8
dodecaedro             20   30   12
icosaedro              12   30   20
cuboctaedro            12   24   14
tetraedro truncado     12   18    8


Achou? Em todas as linhas, $V - A + F$ dá sempre o mesmo número!

**Teorema (Relação de Euler)**
> Em todo poliedro convexo,
> $$V - A + F = 2$$

> 🕵️ **Curiosidade histórica.** O suíço **Leonhard Euler** publicou essa relação em 1758. Ele é um dos matemáticos mais produtivos da história: escreveu sobre praticamente todas as áreas da Matemática da sua época, mesmo tendo perdido a visão nos últimos anos de vida. Há registros de que **Descartes** já conhecia um resultado equivalente mais de cem anos antes, mas suas anotações só foram encontradas e publicadas muito depois.

<details>
<summary>Ver uma ideia da demonstração (o argumento de Cauchy)</summary>

1. Retire uma face do poliedro e "estique" o que sobrou sobre um plano, como se fosse feito de borracha. Você obtém uma rede plana de polígonos com os mesmos $V$ e $A$, mas com uma face a menos. Precisamos mostrar que, nessa rede, $V - A + F = 1$.
2. Divida cada polígono em triângulos, traçando diagonais. Cada diagonal acrescenta 1 aresta e 1 face: $V - A + F$ não muda.
3. Agora vá retirando triângulos pela borda da rede. Se o triângulo tem só uma aresta na borda, você retira 1 aresta e 1 face. Se tem duas arestas na borda, retira 1 vértice, 2 arestas e 1 face. Em nenhum caso $V - A + F$ muda.
4. No fim, sobra um único triângulo: $3 - 3 + 1 = 1$. Devolvendo a face retirada no início, $V - A + F = 2$. ∎
</details>

> ⚠️ **Erro comum.** Achar que a Relação de Euler vale para **qualquer** poliedro. Ela vale para os convexos (e para alguns não convexos), mas **não** para todos. Na moldura, por exemplo, $V - A + F = 16 - 32 + 16 = 0$. O furo muda a conta! E cuidado também com a volta: números que satisfazem $V - A + F = 2$ não garantem que exista um poliedro com eles.

### Contando arestas pelas faces

Cada aresta pertence a exatamente duas faces. Então, se você somar o número de lados de todas as faces, cada aresta é contada **duas vezes**:
$$2A = 3F_3 + 4F_4 + 5F_5 + \cdots$$
em que $F_3$ é o número de faces triangulares, $F_4$ o de quadrangulares, e assim por diante. Do mesmo modo, somando o número de arestas que saem de cada vértice:
$$2A = 3V_3 + 4V_4 + 5V_5 + \cdots$$

Com essas fórmulas e a Relação de Euler, dá para descobrir elementos desconhecidos de um poliedro.

In [6]:
print(f"{'poliedro':<20}{'V':>5}{'A':>5}{'F':>5}{'V − A + F':>12}")
for nome, (vertices, faces) in POLIEDROS_CONVEXOS.items():
    V, A, F = contar_elementos(vertices, faces)
    print(f"{nome:<20}{V:>5}{A:>5}{F:>5}{V - A + F:>12}")

# Um poliedro NÃO convexo, com um furo: a conta muda.
print()
V, A, F = contar_elementos(*u3d.poliedro_exemplo("moldura"))
print(f"{'moldura (com furo)':<20}{V:>5}{A:>5}{F:>5}{V - A + F:>12}")

poliedro                V    A    F   V − A + F
tetraedro               4    6    4           2
cubo                    8   12    6           2
octaedro                6   12    8           2
dodecaedro             20   30   12           2
icosaedro              12   30   20           2
cuboctaedro            12   24   14           2
tetraedro truncado     12   18    8           2

moldura (com furo)     16   32   16           0


### O detetive de poliedros

🕹️ Um poliedro convexo tem faces de alguns tipos. Informe quantas faces de cada tipo ele tem e o "detetive" descobre o número de arestas e de vértices.

In [7]:
CUBOCTAEDRO = u3d.truncar_poliedro(*u3d.poliedro_de_platao("cubo"), 0.5)


def detetive(
    triangulos: int, quadrilateros: int, pentagonos: int, hexagonos: int
) -> None:
    """Descobre A e V de um poliedro convexo a partir das suas faces."""
    F = triangulos + quadrilateros + pentagonos + hexagonos
    soma_dos_lados = 3 * triangulos + 4 * quadrilateros + 5 * pentagonos + 6 * hexagonos
    A = soma_dos_lados / 2  # cada aresta foi contada duas vezes
    V = 2 + A - F  # Relação de Euler

    print(f"F = {triangulos} + {quadrilateros} + {pentagonos} + {hexagonos} = {F}")
    print(
        f"2A = 3·{triangulos} + 4·{quadrilateros} + 5·{pentagonos} + 6·{hexagonos} "
        f"= {soma_dos_lados}"
    )
    if soma_dos_lados % 2 == 1:
        print(
            "A soma dos lados é ímpar: não existe poliedro assim (cada aresta precisa "
            "ser contada duas vezes)."
        )
        return
    print(f"A = {soma_dos_lados} / 2 = {A:g}")
    print(f"V = 2 + A − F = 2 + {A:g} − {F} = {V:g}")
    if V < 4:
        print("Não existe poliedro com menos de 4 vértices.")
        return
    print(
        "Atenção: a conta mostra o que um poliedro assim teria, mas não garante que "
        "ele exista."
    )

    if (triangulos, quadrilateros, pentagonos, hexagonos) == (8, 6, 0, 0):
        # Um poliedro que existe com essas faces: o cuboctaedro (seção 6).
        fig = u3d.criar_figura(titulo="Um exemplo: o cuboctaedro")
        fig = u3d.desenhar_poliedro(fig, *CUBOCTAEDRO, colorir_por_lados=True)
        u3d.mostrar(fig)


def controle(descricao: str, valor: int) -> widgets.IntSlider:
    """Cria um slider de 0 a 20 para uma quantidade de faces."""
    return widgets.IntSlider(
        min=0, max=20, step=1, value=valor, description=descricao,
        continuous_update=False,
    )


interact(
    detetive,
    triangulos=controle("triângulos", 8),
    quadrilateros=controle("quadriláteros", 6),
    pentagonos=controle("pentágonos", 0),
    hexagonos=controle("hexágonos", 0),
);

interactive(children=(IntSlider(value=8, continuous_update=False, description='triângulos', max=20), IntSlider…

**Pare e pense**

1. Um poliedro convexo tem 12 faces pentagonais. Quantas arestas e quantos vértices ele tem?
2. Existe poliedro convexo com 7 faces triangulares e nenhuma outra?

<details>
<summary>Ver respostas</summary>

1. $2A = 5 \cdot 12 = 60$, então $A = 30$; $V = 2 + 30 - 12 = 20$. É o dodecaedro.
2. Não: $3 \cdot 7 = 21$ é ímpar, e a soma dos lados precisa ser $2A$, um número par.
</details>

## 3. Soma dos ângulos de todas as faces

Some os ângulos internos de **todas** as faces de um poliedro. Quanto dá?

Uma face com $n$ lados tem soma dos ângulos internos igual a $(n - 2) \cdot 180°$ (módulo 04). Somando em todas as faces e usando $2A = \sum n$:
$$S = \sum (n - 2) \cdot 180° = 180° \cdot \sum n - 360° \cdot F = 360° \cdot A - 360° \cdot F = 360° (A - F)$$

Pela Relação de Euler, $A - F = V - 2$. Então:

**Teorema**
> Em todo poliedro convexo, a soma dos ângulos de todas as faces é
> $$S = (V - 2) \cdot 360°$$

Repare: a soma só depende do número de **vértices**!

### O Teorema de Descartes, finalmente demonstrado

No notebook 06, você viu que o **defeito** de um vértice é $360°$ menos a soma das faces nele. Somando os defeitos de todos os vértices:
$$\text{soma dos defeitos} = 360° \cdot V - S = 360° \cdot V - (V - 2) \cdot 360° = 720°$$

Esse é o Teorema de Descartes, que antes era só uma curiosidade: agora ele é consequência da Relação de Euler.

In [8]:
print(f"{'poliedro':<20}{'S (face a face)':>17}{'S (fórmula)':>14}{'soma dos defeitos':>20}")
for nome, (vertices, faces) in POLIEDROS_CONVEXOS.items():
    V, A, F = contar_elementos(vertices, faces)
    # Soma face a face: cada face de n lados contribui com (n − 2) · 180°.
    S_faces = 0
    for face in faces:
        S_faces += (len(face) - 2) * 180
    S_formula = (V - 2) * 360
    defeitos = 360 * V - S_faces  # 360° em cada vértice, menos o que as faces ocupam
    print(f"{nome:<20}{S_faces:>16}°{S_formula:>13}°{defeitos:>19}°")

poliedro              S (face a face)   S (fórmula)   soma dos defeitos
tetraedro                        720°          720°                720°
cubo                            2160°         2160°                720°
octaedro                        1440°         1440°                720°
dodecaedro                      6480°         6480°                720°
icosaedro                       3600°         3600°                720°
cuboctaedro                     3600°         3600°                720°
tetraedro truncado              3600°         3600°                720°


## 4. Planificações

> 📌 **Definição.** Uma **planificação** de um poliedro é uma figura plana formada pelas suas faces, ligadas pelas arestas, que pode ser **dobrada** para montar o poliedro.

🎯 **No dia a dia:** toda caixa de papelão nasce como uma planificação (com algumas abas extras para colar). Embalagens de leite, de pizza, de presente; brinquedos de papel (*papercraft*); e até satélites com painéis dobráveis usam essa ideia.

🕹️ Escolha um sólido e dobre a sua planificação.

In [9]:
SOLIDOS_DE_PLATAO = ["tetraedro", "cubo", "octaedro", "dodecaedro", "icosaedro"]


def dobrar_solido(solido: str, dobra: float) -> None:
    """Desenha a planificação do sólido dobrada até a fração escolhida."""
    planificacao = u3d.planificacao_padrao(solido)
    faces = u3d.dobrar_planificacao(planificacao, dobra)
    fig = u3d.criar_figura(titulo=f"Planificação do {solido}")
    fig = u3d.desenhar_faces(fig, faces)
    if dobra == 0:
        fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))  # vista de cima
    u3d.mostrar(fig)
    print(f"Faces: {len(faces)}")
    if dobra == 1:
        print("Montado!")


interact(
    dobrar_solido,
    solido=widgets.Dropdown(options=SOLIDOS_DE_PLATAO, description="sólido"),
    dobra=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0.5, description="Dobrar",
        continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='sólido', options=('tetraedro', 'cubo', 'octaedro', 'dodecaedro', '…

### Toda figura de 6 quadrados fecha um cubo?

Não! O cubo tem exatamente **11** planificações diferentes (sem contar as que são apenas giradas ou espelhadas). 

🕹️ Escolha uma figura de 6 quadrados, tente prever se ela fecha um cubo e depois dobre para conferir.

> 💡 **Dica.** Ao imaginar a dobra, escolha um quadrado para ser a "base" e pense em quem vira a face de cima. Se dois quadrados disputarem o mesmo lugar, a figura não é planificação.

In [10]:
# Cada figura é uma lista de casas (coluna, linha) de um quadriculado.
PLANIFICACOES_DO_CUBO = {
    "Cruz": [(1, 0), (0, 1), (1, 1), (2, 1), (3, 1), (1, 2)],
    "Bloco 2 × 3": [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1)],
    "Escada": [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (3, 2)],
    "Pontas do mesmo lado": [(0, 0), (1, 0), (2, 0), (3, 0), (0, 1), (3, 1)],
    "Pontas de lados opostos": [(0, 0), (1, 0), (2, 0), (3, 0), (0, 1), (3, -1)],
    "Fila de 6": [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0)],
}
# Uma cor para cada um dos seis quadrados.
CORES_DOS_QUADRADOS = [
    u3d.CORES["plano"], u3d.CORES["plano_secundario"], u3d.CORES["plano_terciario"],
    u3d.CORES["solido"], u3d.CORES["destaque"], u3d.CORES["reta_secundaria"],
]


def dobrar_quadrados(figura: str, dobra: float, mostrar_resposta: bool) -> None:
    """Dobra uma figura de 6 quadrados e, se pedido, diz se ela fecha um cubo."""
    planificacao = u3d.planificacao_de_quadrados(PLANIFICACOES_DO_CUBO[figura])
    faces = u3d.dobrar_planificacao(planificacao, dobra)
    fig = u3d.criar_figura(titulo=figura)
    fig = u3d.desenhar_faces(fig, faces, cores=CORES_DOS_QUADRADOS)
    if dobra == 0:
        fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))  # vista de cima
    u3d.mostrar(fig)
    if mostrar_resposta:
        print(f"Fecha um cubo? {u3d.planificacao_fecha(planificacao)}")


interact(
    dobrar_quadrados,
    figura=widgets.Dropdown(options=list(PLANIFICACOES_DO_CUBO), description="figura"),
    dobra=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0, description="Dobrar",
        continuous_update=False,
    ),
    mostrar_resposta=widgets.Checkbox(value=False, description="Mostrar resposta"),
);

interactive(children=(Dropdown(description='figura', options=('Cruz', 'Bloco 2 × 3', 'Escada', 'Pontas do mesm…

**Pare e pense**

1. Por que uma fila de 6 quadrados não fecha um cubo?
2. Na planificação em cruz, qual quadrado fica oposto ao quadrado do centro da cruz?

<details>
<summary>Ver respostas</summary>

1. Os quatro primeiros quadrados dão a volta e formam as "paredes"; o quinto cai em cima do primeiro, e o sexto em cima do segundo. Sobra a tampa e o fundo sem cobertura.
2. O quadrado da ponta da fila, separado do centro por um quadrado.
</details>

## 5. Poliedros de Platão e poliedros regulares

> 📌 **Definição.** Um poliedro é **de Platão** quando:
> 1. todas as faces têm o **mesmo número $n$** de arestas;
> 2. todos os vértices têm o **mesmo número $m$** de arestas;
> 3. vale a Relação de Euler.

> 📌 **Definição.** Um poliedro convexo é **regular** quando todas as faces são **polígonos regulares congruentes** e todos os ângulos poliédricos são **congruentes**.

> ⚠️ **Erro comum.** Achar que "de Platão" e "regular" são a mesma coisa. Todo poliedro regular é de Platão, mas não o contrário. Uma **caixa de sapatos** é de Platão (todas as faces têm 4 lados, todos os vértices têm 3 arestas), mas não é regular: as faces são retângulos de tamanhos diferentes.

### Teorema 1: só existem cinco classes de poliedros de Platão

Contando as arestas pelas faces e pelos vértices: $2A = nF$ e $2A = mV$. Então $F = \dfrac{2A}{n}$ e $V = \dfrac{2A}{m}$. Substituindo na Relação de Euler:
$$\frac{2A}{m} - A + \frac{2A}{n} = 2 \quad\Longrightarrow\quad \frac{1}{m} + \frac{1}{n} = \frac{1}{2} + \frac{1}{A}$$

Como $\frac{1}{A} > 0$, precisamos de $\dfrac{1}{m} + \dfrac{1}{n} > \dfrac{1}{2}$, com $m \ge 3$ e $n \ge 3$. Isso só acontece em **cinco** casos. Vamos deixar o computador encontrá-los.

In [11]:
print(" n   m     A    F    V")
for n in range(3, 11):  # lados de cada face
    for m in range(3, 11):  # arestas em cada vértice
        if 1 / m + 1 / n > 1 / 2:
            A = 1 / (1 / m + 1 / n - 1 / 2)
            F = 2 * A / n
            V = 2 * A / m
            print(f"{n:2d}  {m:2d}  {A:4.0f} {F:4.0f} {V:4.0f}")

# O laço vai até 10 só por praticidade: para n ou m maiores, a soma 1/m + 1/n fica
# ainda menor, e a condição 1/m + 1/n > 1/2 continua falhando.

 n   m     A    F    V
 3   3     6    4    4
 3   4    12    8    6
 3   5    30   20   12
 4   3    12    6    8
 5   3    30   12   20


### Teorema 2: só existem cinco poliedros regulares

No notebook 06, você viu que, num vértice de um ângulo poliédrico convexo, só cabem 3, 4 ou 5 triângulos equiláteros, 3 quadrados ou 3 pentágonos regulares. Cada uma dessas cinco possibilidades dá exatamente um poliedro regular:

| Vértice | Poliedro regular | $n$ | $m$ | $F$ | $A$ | $V$ |
|---|---|---|---|---|---|---|
| 3 triângulos | tetraedro | 3 | 3 | 4 | 6 | 4 |
| 4 triângulos | octaedro | 3 | 4 | 8 | 12 | 6 |
| 5 triângulos | icosaedro | 3 | 5 | 20 | 30 | 12 |
| 3 quadrados | cubo (hexaedro) | 4 | 3 | 6 | 12 | 8 |
| 3 pentágonos | dodecaedro | 5 | 3 | 12 | 30 | 20 |

É por isso que os dados justos do gancho só têm 4, 6, 8, 12 e 20 faces!

> 🕵️ **Curiosidade histórica.** O filósofo grego **Platão** (por volta de 400 a.C.) associou quatro desses sólidos aos "elementos" da natureza: fogo (tetraedro), terra (cubo), ar (octaedro) e água (icosaedro); o dodecaedro representaria o universo. Por volta de 300 a.C., **Euclides** encerrou *Os Elementos* demonstrando que só existem cinco poliedros regulares. Quase dois mil anos depois, **Johannes Kepler** tentou explicar as distâncias entre os planetas encaixando os cinco sólidos uns dentro dos outros. A ideia estava errada, mas ajudou Kepler a mergulhar nos dados astronômicos que o levaram às suas famosas leis do movimento dos planetas.

🎯 **Onde eles aparecem hoje**
- **Química:** a molécula de metano tem a forma de um tetraedro (o carbono no centro e os hidrogênios nos vértices).
- **Cristais:** o sal de cozinha forma cristais cúbicos; o diamante costuma formar cristais octaédricos.
- **Biologia:** a "cápsula" de proteína de muitos vírus tem simetria de icosaedro.
- **Jogos:** os dados de RPG.

🕹️ Compare os cinco sólidos abaixo.

In [12]:
def comparar_solidos(solido: str, mostrar_angulo_poliedrico: bool) -> None:
    """Desenha um sólido de Platão e mostra os seus números."""
    vertices, faces = u3d.poliedro_de_platao(solido)
    fig = u3d.criar_figura(titulo=solido.capitalize())
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.45)
    if mostrar_angulo_poliedrico:
        # As faces que se encontram no vértice 0 formam um ângulo poliédrico.
        for face in faces:
            if 0 in face:
                fig = u3d.desenhar_poligono(
                    fig, [vertices[i] for i in face], cor=u3d.CORES["destaque"],
                    opacidade=0.8,
                )
        fig = u3d.desenhar_ponto(fig, vertices[0], rotulo="V")
    u3d.mostrar(fig)

    V, A, F = contar_elementos(vertices, faces)
    n = len(faces[0])  # lados de cada face (todas são iguais)
    m = sum(1 for face in faces if 0 in face)  # faces (e arestas) no vértice 0
    diedros = list(u3d.diedros_do_poliedro(vertices, faces).values())
    angulo_interno = 180 * (n - 2) / n  # ângulo interno do polígono regular
    soma_no_vertice = m * angulo_interno
    defeito = 360 - soma_no_vertice

    print(f"V = {V}, A = {A}, F = {F}, n = {n}, m = {m}")
    print(
        f"Diedros (conferidos com o utils_3d): o menor mede {min(diedros):.1f}° e o "
        f"maior, {max(diedros):.1f}°. Todos são iguais."
    )
    print(f"Soma das faces num vértice: {m} × {angulo_interno:g}° = {soma_no_vertice:g}°")
    print(f"Defeito: 360° − {soma_no_vertice:g}° = {defeito:g}°")


interact(
    comparar_solidos,
    solido=widgets.Dropdown(options=SOLIDOS_DE_PLATAO, description="sólido"),
    mostrar_angulo_poliedrico=widgets.Checkbox(
        value=False, description="Mostrar ângulo poliédrico de um vértice"
    ),
);

interactive(children=(Dropdown(description='sólido', options=('tetraedro', 'cubo', 'octaedro', 'dodecaedro', '…

## 6. Além de Platão: duais e sólidos de Arquimedes

### Poliedros duais

> 📌 **Definição.** O **dual** de um poliedro é obtido marcando um ponto no centro de cada face e ligando os centros de faces vizinhas.

Cada face vira um vértice, e cada vértice vira uma face: no dual, os números $V$ e $F$ trocam de lugar, e $A$ continua igual.
- O dual do **cubo** é o **octaedro** (e vice-versa).
- O dual do **dodecaedro** é o **icosaedro** (e vice-versa).
- O dual do **tetraedro** é outro **tetraedro**.

### Sólidos de Arquimedes

Se você permitir faces regulares de **dois ou mais tipos**, mas exigir que todos os vértices sejam iguais, aparecem os **sólidos de Arquimedes**. Além das famílias infinitas de prismas e antiprismas, existem exatamente **13** deles.

Um jeito de obter vários: **truncar** (cortar os cantos) de um sólido de Platão.
- Truncando o icosaedro a $\frac{1}{3}$ de cada aresta, você obtém o **icosaedro truncado**: 12 pentágonos e 20 hexágonos. É o formato clássico da **bola de futebol**!
- Truncando o cubo até a metade das arestas, você obtém o **cuboctaedro**: 8 triângulos e 6 quadrados.

> 📎 **Conexão.** Em 1985, químicos descobriram uma molécula com 60 átomos de carbono exatamente nos vértices de um icosaedro truncado: o **buckminsterfulereno** (C₆₀). A descoberta rendeu o Prêmio Nobel de Química de 1996.

🕹️ Escolha um sólido e uma operação. Repare que a Relação de Euler continua valendo em todos os resultados.

In [ ]:
NOMES_DOS_POLIGONOS = {
    3: "triângulos", 4: "quadriláteros", 5: "pentágonos", 6: "hexágonos",
    8: "octógonos", 10: "decágonos",
}


def transformar_solido(solido: str, operacao: str, corte: float) -> None:
    """Desenha o sólido, o seu dual ou o sólido truncado, e confere Euler."""
    # Com o icosaedro e corte = 1/3, os hexágonos ficam regulares (bola de futebol);
    # o slider anda de 0,05 em 0,05, então o valor mais próximo é 0,35. Com o cubo e
    # corte = 0,5, aparece o cuboctaedro.
    corte = round(corte, 2)  # o slider pode devolver 0.35000000000000003
    vertices, faces = u3d.poliedro_de_platao(solido)
    fig = u3d.criar_figura(titulo=f"{solido.capitalize()}: {operacao.lower()}")
    if operacao == "Nenhuma":
        novo = (vertices, faces)
    elif operacao == "Dual":
        # O original, bem transparente, com o dual por dentro.
        fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.15)
        novo = u3d.poliedro_dual(vertices, faces)
    else:
        novo = u3d.truncar_poliedro(vertices, faces, corte)
    fig = u3d.desenhar_poliedro(fig, *novo, colorir_por_lados=True)
    u3d.mostrar(fig)

    V, A, F = contar_elementos(*novo)
    print(f"V = {V}, A = {A}, F = {F}")
    print(f"V − A + F = {V} − {A} + {F} = {V - A + F}")
    # Quantas faces de cada tipo: um dicionário "número de lados → quantidade".
    tipos = {}
    for face in novo[1]:
        tipos[len(face)] = tipos.get(len(face), 0) + 1
    print("Faces:")
    for lados, quantidade in sorted(tipos.items()):
        print(f"  {quantidade} {NOMES_DOS_POLIGONOS.get(lados, f'polígonos de {lados} lados')}")


interact(
    transformar_solido,
    solido=widgets.Dropdown(options=SOLIDOS_DE_PLATAO, description="sólido"),
    operacao=widgets.Dropdown(
        options=["Nenhuma", "Dual", "Truncar"], description="operação"
    ),
    corte=widgets.FloatSlider(
        min=0.05, max=0.5, step=0.05, value=0.35, description="corte",
        continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='sólido', options=('tetraedro', 'cubo', 'octaedro', 'dodecaedro', '…

**Pare e pense**

1. O dual do dual do cubo é o quê?
2. A bola de futebol tem 12 pentágonos e 20 hexágonos. Quantas arestas e quantos vértices ela tem? Confira no widget.

<details>
<summary>Ver respostas</summary>

1. Um cubo (menor que o original).
2. $2A = 5 \cdot 12 + 6 \cdot 20 = 180$, então $A = 90$; $F = 32$; $V = 2 + 90 - 32 = 60$.
</details>

## Resumo

**Poliedro convexo:** o plano de cada face deixa todo o poliedro de um mesmo lado. Elementos: $V$ vértices, $A$ arestas, $F$ faces.

**Relação de Euler** (poliedros convexos): $V - A + F = 2$. Não vale para todos os não convexos (moldura: $V - A + F = 0$).

**Contagem de arestas:** $2A = 3F_3 + 4F_4 + 5F_5 + \cdots = 3V_3 + 4V_4 + 5V_5 + \cdots$

**Soma dos ângulos das faces:** $S = (V - 2) \cdot 360°$. **Descartes:** a soma dos defeitos dos vértices é $720°$.

**Planificação:** figura plana que, dobrada, forma o poliedro. O cubo tem 11 planificações.

**Poliedro de Platão:** faces com o mesmo $n$, vértices com o mesmo $m$, Euler válido. Só cinco classes: $(n, m) = (3,3), (3,4), (3,5), (4,3), (5,3)$.

**Poliedro regular:** faces regulares congruentes e ângulos poliédricos congruentes. Só cinco: tetraedro, octaedro, icosaedro, cubo, dodecaedro. Todo regular é de Platão; nem todo de Platão é regular.

**Duais:** trocam $V$ e $F$. Cubo ↔ octaedro; dodecaedro ↔ icosaedro; tetraedro ↔ tetraedro.

**Sólidos de Arquimedes:** faces regulares de mais de um tipo, vértices iguais. Treze, entre eles o icosaedro truncado (bola de futebol) e o cuboctaedro.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**.

O gabarito comentado está em [`0607b_poliedros_convexos_gabarito.ipynb`](0607b_poliedros_convexos_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Todo poliedro convexo satisfaz V − A + F = 2.
# (b) Todo poliedro que satisfaz V − A + F = 2 é convexo.
# (c) Todo poliedro de Platão é regular.
# (d) Existem exatamente cinco poliedros regulares.
# (e) O dual do cubo é o octaedro.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": False, "d": True, "e": True}
secoes_ex1 = {
    "a": "seção 2 - A Relação de Euler",
    "b": "seção 2 - A Relação de Euler",
    "c": "seção 5 - Poliedros de Platão e poliedros regulares",
    "d": "seção 5 - Poliedros de Platão e poliedros regulares",
    "e": "seção 6 - Além de Platão: duais e sólidos de Arquimedes",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; releia o erro comum "
        f"da seção 2 e as definições ({secoes_ex1[item]})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: Relação de Euler. Responda com números inteiros.
# (a) Um poliedro convexo tem 8 vértices e 12 arestas. Quantas faces?
# (b) Um poliedro convexo tem 20 faces triangulares. Quantas arestas e quantos
#     vértices?
# (c) Um poliedro convexo tem 12 faces pentagonais. Quantas arestas e quantos
#     vértices?
faces_ex2a = ...
arestas_ex2b = ...
vertices_ex2b = ...
arestas_ex2c = ...
vertices_ex2c = ...

print(faces_ex2a, arestas_ex2b, vertices_ex2b, arestas_ex2c, vertices_ex2c)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", "faces", faces_ex2a, 6),
    ("b", "arestas", arestas_ex2b, 30),
    ("b", "vértices", vertices_ex2b, 12),
    ("c", "arestas", arestas_ex2c, 30),
    ("c", "vértices", vertices_ex2c, 20),
]
for item, elemento, resposta, certa in respostas_ex2:
    assert resposta is not ... and resposta == certa, (
        f"Tente de novo, revise: no item ({item}), confira as {elemento}; use 2A = "
        "(soma dos lados de todas as faces) e depois V − A + F = 2 (seção 2 - A "
        "Relação de Euler)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: soma dos ângulos de todas as faces, em graus.
# (a) cubo; (b) tetraedro; (c) icosaedro.
soma_ex3 = {
    "a": ...,
    "b": ...,
    "c": ...,
}

print(soma_ex3)

In [ ]:
# Verificação
gabarito_ex3 = {"a": 2160, "b": 720, "c": 3600}
for item, certa in gabarito_ex3.items():
    resposta = soma_ex3[item]
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), S = (V − 2) · 360°; comece "
        "contando os vértices (seção 3 - Soma dos ângulos de todas as faces)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a bola de futebol. Ela tem 12 faces pentagonais e 20
# hexagonais. Calcule, em Python:
# (a) o número de faces F;
# (b) o número de arestas A;
# (c) o número de vértices V;
# (d) quantas arestas saem de cada vértice, sabendo que todos os vértices são
#     iguais.
pentagonos = 12
hexagonos = 20

F_ex4 = ...
A_ex4 = ...
V_ex4 = ...
m_ex4 = ...

print(F_ex4, A_ex4, V_ex4, m_ex4)

In [ ]:
# Verificação
respostas_ex4 = [
    ("a", F_ex4, 32, "some as faces dos dois tipos"),
    ("b", A_ex4, 90, "use 2A = (soma dos lados de todas as faces)"),
    ("c", V_ex4, 60, "use V − A + F = 2"),
    ("d", m_ex4, 3, "para (d), use 2A = m · V"),
]
for item, resposta, certa, dica in respostas_ex4:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), {dica} (seção 2 - A Relação de "
        "Euler)."
    )
print("Certo!")

# Conferência com o utils_3d: truncando o icosaedro a 1/3 de cada aresta.
bola = u3d.truncar_poliedro(*u3d.poliedro_de_platao("icosaedro"), 1 / 3)
V, A, F = contar_elementos(*bola)
print(f"Icosaedro truncado: V = {V}, A = {A}, F = {F}")
fig = u3d.criar_figura(titulo="A bola de futebol (icosaedro truncado)")
fig = u3d.desenhar_poliedro(fig, *bola, colorir_por_lados=True)
u3d.mostrar(fig)

In [ ]:
# Exercício Intermediário 5: um poliedro convexo tem 3 faces triangulares, 1
# quadrangular, 1 pentagonal e 2 hexagonais. Quantos vértices ele tem?
V_ex5 = ...

print(V_ex5)

In [ ]:
# Verificação
assert V_ex5 is not ... and abs(V_ex5 - 10) <= 0.01, (
    "Tente de novo, revise: 2A = 3·3 + 4·1 + 5·1 + 6·2; depois use a Relação de "
    "Euler com F = 7 (seção 2 - A Relação de Euler)."
)
print("Certo!")

In [14]:
# Figuras de apoio do exercício 6: quatro figuras de 6 quadrados, vistas de cima.
FIGURAS_EX6 = {
    "a": "Cruz",
    "b": "Bloco 2 × 3",
    "c": "Escada",
    "d": "Pontas do mesmo lado",
}
for item, figura in FIGURAS_EX6.items():
    planificacao = u3d.planificacao_de_quadrados(PLANIFICACOES_DO_CUBO[figura])
    fig = u3d.criar_figura(titulo=f"({item}) {figura}", altura=320)
    fig = u3d.desenhar_faces(
        fig, u3d.dobrar_planificacao(planificacao, 0), cores=CORES_DOS_QUADRADOS
    )
    fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))
    u3d.mostrar(fig)

In [ ]:
# Exercício Intermediário 6: quais das quatro figuras acima fecham um cubo?
# Responda com uma lista de quatro True ou False, na ordem (a), (b), (c), (d).
fecha_ex6 = [..., ..., ..., ...]

print(fecha_ex6)

In [ ]:
# Verificação
assert fecha_ex6 == [True, False, True, False], (
    "Tente de novo, revise: imagine a dobra escolhendo um quadrado como base; se "
    "dois quadrados disputarem o mesmo lugar, não fecha (seção 4 - Planificações)."
)
print("Certo!")

# Conferência com o utils_3d e a dobra completa das quatro figuras.
for item, figura in FIGURAS_EX6.items():
    planificacao = u3d.planificacao_de_quadrados(PLANIFICACOES_DO_CUBO[figura])
    print(f"({item}) {figura}: fecha? {u3d.planificacao_fecha(planificacao)}")
    fig = u3d.criar_figura(titulo=f"({item}) {figura}, dobrada", altura=320)
    fig = u3d.desenhar_faces(
        fig, u3d.dobrar_planificacao(planificacao, 1), cores=CORES_DOS_QUADRADOS
    )
    u3d.mostrar(fig)

### Desafio

In [ ]:
# Exercício Desafio 7: caçando os poliedros de Platão. Encontre TODOS os pares
# (n, m), com 3 ≤ n ≤ 20 e 3 ≤ m ≤ 20, que satisfazem 1/m + 1/n > 1/2, e guarde
# cada par (n, m) na lista pares (use pares.append((n, m))).
pares = []
for n in range(3, 21):
    for m in range(3, 21):
        ...  # complete aqui

print(pares)

In [ ]:
# Verificação
assert pares is not ... and set(pares) == {(3, 3), (3, 4), (4, 3), (3, 5), (5, 3)}, (
    "Tente de novo, revise: use dois laços for, um dentro do outro, e um if com a "
    "condição (seção 5 - Poliedros de Platão e poliedros regulares)."
)
print("Certo!")

In [ ]:
# Exercício Desafio 8: Euler e o furo. Use a função contar_elementos (seção 1) para
# calcular V − A + F na moldura.
vertices_ex8, faces_ex8 = u3d.poliedro_exemplo("moldura")

euler_ex8 = ...

print(euler_ex8)

In [ ]:
# Verificação
assert euler_ex8 is not ... and euler_ex8 == 0, (
    "Tente de novo, revise: conte V, A e F com a função da seção 1 e faça a conta "
    "(seção 2 - A Relação de Euler)."
)
print("Certo!")
print(
    'Para poliedros com "furos", a conta muda: com um furo dá 0, com dois furos dá '
    "−2, e assim por diante. Esse número, chamado característica de Euler, é um dos "
    'pontos de partida da Topologia, a área da Matemática que estuda as formas "de '
    'borracha".'
)

## Conexões

**Isso depende de:**
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (diedros entre faces)
- [`0606a_triedros_e_angulos_poliedricos.ipynb`](0606a_triedros_e_angulos_poliedricos.ipynb) (ângulos poliédricos convexos, polígonos regulares num vértice, defeito angular)
- [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb) (soma dos ângulos internos de um polígono)
- [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e seus ângulos internos)

**Isso será usado em:**
- [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) (o prisma como poliedro, planificações)
- 📌 Pirâmide (a pirâmide como poliedro, tetraedro regular)
- 📌 Inscrição e circunscrição: esfera e poliedros (esferas inscritas e circunscritas aos poliedros regulares)
- 📌 Módulo 12 — Matemática Discreta (grafos planares e a Relação de Euler para grafos)